In [1]:
import numpy as np
from numpy.random import uniform
from scipy.optimize import minimize
from scipy.stats import qmc
from scipy.stats import norm
import cyipopt
import warnings
import collections
from scipy import linalg
from smt.applications.ego import Evaluator
from smt.surrogate_models import KRG
from smt.design_space import DesignSpace 
from smt.problems.problem import Problem as SMTProblem

#### Gaussian Process SMT

In [2]:
class GaussianProcess:
    def __init__(self, ndim, xlimits=None):
        self.ndim = ndim
        self.xlimits = xlimits
        self.training_x = []
        self.training_y = []
        self.trained = False
    
    # Abstract method for computing the mean of the GP at a given input x
    def mean(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the GP mean

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Mean of GP at x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method mean")
    
    # Abstract method for computing the covariance of the GP at a given input x
    def covariance(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the GP covariance

        Parameters
        ---------
        x: ndarray[n, nx]

        Returns
        -------
        ndarray[n, n]
           Covariance of GP at w.r.t. x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method covariance")

    # Abstract method for computing the variance of the GP at a given input x
    def variance(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the GP variance

        Parameters
        ---------
        x: ndarray[n, nx]

        Returns
        ------
        ndarray[n, 1]
           Variance of GP at x
        """
        y = np.ndarray((self.ndim, 1))
        for i in range(x.shape[1]):
            y[i][0] = covariance(np.atleast_2d(x[i,:]))[0][0]
        return y
        #return np.atleast_2d(np.diag(covariance(x))).T

    # Retrieves the bounds of the input space if xlimits is provided.
    def get_bounds(self):
        if self.xlimits is None:
            return None
        else:
            return [(self.xlimits[i][0], self.xlimits[i][1]) for i in range(self.ndim)]

    # Abstract method for training the GP
    def train(self, x: np.ndarray, y: np.ndarray) -> np.ndarray:
        """
        train the GP model

        Parameters
        ---------
        x : ndarray[n, nx]
        y : ndarray[n, 1]

        """
        NotImplementedError("Child class of GaussianProcess should implement method train")

    # Abstract method for computing the gradient of the mean of the GP at a given input x
    def mean_gradient(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the gradien of GP mean

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Gradient of Mean of GP at x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method mean_gradient")
    
    # Abstract method for computing the gradient of variance of the GP at a given input x
    def variance_gradient(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the gradient of GP variance

        Parameters
        ---------
        x: ndarray[n, nx]

        Returns
        -------
        ndarray[n, n]
           Gradient of variance of GP at w.r.t. x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method variance_gradient")


class smtKRG(GaussianProcess):
    def __init__(self, theta, xlimits, ndim, corr="squar_exp", noise0=None, random_state=None):
        super().__init__(ndim, xlimits)
        if random_state is None:
            random_state = 42
        design_space = DesignSpace(xlimits, random_state=random_state)
        if noise0 is None:
            self.surrogatesmt = KRG(design_space=design_space,
                             print_global=False,
                             eval_noise=False,
                             corr=corr)
        else:
            self.surrogatesmt = KRG(design_space=design_space,
                             print_global=False,
                             noise0=noise0,
                             eval_noise=False,
                             corr=corr)
        self.trained = False

    def mean(self, x):
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict mean or variances")
        return self.surrogatesmt.predict_values(x)

    def variance(self, x):
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict mean or variances")
        return self.surrogatesmt.predict_variances(x)

    def train(self, x, y):
        self.training_x = x
        self.training_y = y
        self.surrogatesmt.set_training_values(x, y)
        self.surrogatesmt.train()
        self.trained = True

    def mean_gradient(self, x: np.ndarray) -> np.ndarray:
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict gradient")
        assert (np.size(x,-1) == self.ndim)
        gradient = [
            self.surrogatesmt._predict_derivatives(x, kx) for kx in range(self.ndim) 
        ]
        return np.atleast_2d(gradient).T

    def variance_gradient(self, x: np.ndarray) -> np.ndarray:
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict gradient")
        return self.surrogatesmt.predict_variance_gradient(x)


#### Acquisition Functions

In [3]:
# A base class for acquisition functions
class acquisition(object):
    def __init__(self, gpsurrogate):
        assert isinstance(gpsurrogate, GaussianProcess) # add something here
        self.gpsurrogate = gpsurrogate
        self.has_gradient = False
    
    # Abstract method to evaluate the acquisition function at x.
    def evaluate(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError("Child class of acquisition should implement method evaluate")

    # Abstract method to evaluate the gradient of acquisition function at x.
    def eval_g(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError("Child class of acquisition should implement method evaluate")

# A subclass of acquisition, implementing the Lower Confidence Bound (LCB) acquisition function.
class LCBacquisition(acquisition):
    def __init__(self, gpsurrogate, beta=3.0):
        # For BnB, we need to extract this beta eventually 
        super().__init__(gpsurrogate)
        self.beta = beta
        self.has_gradient = True

    # Method to evaluate the acquisition function at x.
    def evaluate(self, x : np.ndarray) -> np.ndarray:
        mu = self.gpsurrogate.mean(x)
        sig2 = self.gpsurrogate.variance(x)
        return mu - self.beta * np.sqrt(sig2)

    def eval_g(self, x: np.ndarray) -> np.ndarray:
        mu = self.gpsurrogate.mean(x)
        sig2 = self.gpsurrogate.variance(x)
        dsig2_dx = self.gpsurrogate.variance_gradient(x)
        dmu_dx = self.gpsurrogate.mean_gradient(x)
        return dmu_dx - 0.5 * self.beta * dsig2_dx / np.sqrt(sig2)

# A subclass of acquisition, implementing the Expected improvement (EI) acquisition function.
class EIacquisition(acquisition):
    def __init__(self, gpsurrogate):
        super().__init__(gpsurrogate)
        self.has_gradient = True

    # Method to evaluate the acquisition function at x.
    def evaluate(self, x : np.ndarray) -> np.ndarray:        
        y_data = self.gpsurrogate.training_y
        y_min = y_data[np.argmin(y_data[:, 0])]

        pred = self.gpsurrogate.mean(x)
        sig = np.sqrt(self.gpsurrogate.variance(x))

        retval = []
        if sig.size == 1 and np.abs(sig) > 1e-12:
            z = (y_min - pred) / sig
            retval = (y_min - pred) * norm.cdf(z) + sig * norm.pdf(z)
            retval *= -1.
        elif sig.size == 1 and np.abs(sig) <= 1e-12:
            retval = 0.0
        elif sig.size > 1:
            raise NotImplementedError("TODO --- Not implemented yet!")

        return retval

    def eval_g(self, x: np.ndarray) -> np.ndarray:
        y_data = self.gpsurrogate.training_y
        y_min = y_data[np.argmin(y_data[:, 0])]

        mean = self.gpsurrogate.mean(x)
        sig2 = self.gpsurrogate.variance(x)
        sig = np.sqrt(sig2)

        grad_EI = None
        if sig.size == 1 and np.abs(sig) > 1e-12:
            dmean_dx = self.gpsurrogate.mean_gradient(x)
            dsig2_dx = self.gpsurrogate.variance_gradient(x)
            dsig_dx = 0.5 * dsig2_dx / sig

            z = (y_min - mean) / sig
            ncdf = norm.cdf(z)
            npdf = norm.pdf(z)
            EI = (y_min - mean) * ncdf + sig * npdf

            dz_dx = -dmean_dx / sig - (y_min - mean) * dsig_dx / sig**2         
            grad_EI = -dmean_dx * ncdf + dsig_dx * npdf
            grad_EI *= -1.
        elif sig.size == 1 and np.abs(sig) <= 1e-12:
            grad_EI = 0.0
        elif sig.size > 1:
            raise NotImplementedError("TODO --- Not implemented yet!")

        return grad_EI

#### Problem

In [4]:
def check_required_keys(user_dict, required_keys):
    for key in required_keys:
        if key not in user_dict:
            raise KeyError(f"Missing required key: '{key}'")

In [5]:
class Problem:
    def __init__(self, ndim, xlimits, name=" ", constraints=[]):
        self.ndim = ndim
        self.xlimits = xlimits
        assert self.xlimits.shape[0] == ndim            
        assert isinstance(name, str)
        assert isinstance(constraints, collections.abc.Sequence)
        self.name = name
        self.sampler = qmc.LatinHypercube(ndim)
        self.constraints = constraints
            
    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        """
        problem evaluation y = f(x) of
        a scalar valued function f

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Function values
        """
        raise NotImplementedError("Child class of hiopProblem should implement method _evaluate")

    def evaluate(self, x: np.ndarray) -> np.ndarray:
        """
        problem callback y = f(x) of
        the scalar valued function  f

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Function values (cast to reals)
        """
        y = np.ndarray((x.shape[0], 1))
        y[:,:] = self._evaluate(x)
        return y

    def sample(self, nsample: int) -> np.ndarray:
        """
        generate nsample samples from domain defined
        by xlimits

        Parameters
        -------
        nsample : int

        Returns
        -------
        ndarray[nsample, nx]
           Samples from domain defined by xlimits
        """

        # uniform
        # xsample = np.zeros((nsample, self.ndim))
        # for j in range(self.ndim):
        #    xsample[:, j] = uniform(self.xlimits[j][0], self.xlimits[j][1], size=nsample)

        # from predefined sampler
        xsample = self.sampler.random(nsample)
        xsample = self.xlimits[:,0] + (self.xlimits[:,1] - self.xlimits[:,0]) * xsample

        return xsample

    def set_constraints(self, constraints):
        self.constraints = constraints

class LpNormProblem(Problem):
    def __init__(self, ndim, xlimits, p=2.0, constraints=[]):
        name = "LpNormProblem"
        super().__init__(ndim, xlimits, name=name, constraints=constraints)
        self.p = p

    def _evaluate(self, x):
        ne, nx = x.shape
        assert nx == self.ndim
        y = np.zeros((ne, 1))
        ytemp = np.linalg.norm(x, ord=self.p, axis=1)
        if len(ytemp.shape) == 1:
            y[:,0] = ytemp[:]
        elif len(ytemp.shape) == 2:
            y[:,:] = ytemp[:,:]
        return y

class BraninProblem(Problem):
    def __init__(self, constraints=[]):
        ndim = 2
        xlimits = np.array([[-5.0, 10], [0.0, 15]]) 
        name = 'Branin'
        super().__init__(ndim, xlimits, name=name, constraints=constraints)
            
    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        
        ne, nx = x.shape
        assert nx == self.ndim
        
        y = np.zeros((ne, 1), complex)
        b = 5.1 / (4.0 * (np.pi) ** 2)
        c = 5.0 / np.pi
        r = 6.0
        s = 10.0
        t = 1.0 / (8.0 * np.pi)
        
        arg1 = (x[:,1] - b * x[:,0]**2 + c * x[:,0] - r)
        y[:,0] = arg1**2 + s * (1 - t) * np.cos(x[:,0]) + s
        
        return y


In [6]:
from typing import Callable, Dict, List, Union, Tuple
from scipy.optimize import NonlinearConstraint

class IpoptProb:
    def __init__(self, objective, gradient, constraint:Union[Dict, List[Dict]], xbounds, solver_options=None):
        self.cons = constraint
        self.eval_f = objective
        self.eval_g  = gradient
        self.xl = [b[0] for b in xbounds]
        self.xu = [b[1] for b in xbounds]
        self.cl = []
        self.cu = []
        self.nvar = len(xbounds)
        
        self.ipopt_options = solver_options

        if isinstance(self.cons, list):
            # constraints is provided as a list of dict, supported by SLSQP and Ipopt
            for con in self.cons:
                check_required_keys(con,['type', 'fun'])
                if con['type'] == 'eq':
                    self.cl.append(0.0)
                    self.cu.append(0.0)
                elif con['type'] == 'ineq':
                    self.cl.append(0.0)
                    self.cu.append(np.inf)
                else:
                    raise ValueError(f"Unknown constraint type: {con['type']}")
        elif isinstance(self.cons, dict):
            check_required_keys(self.cons,['cons', 'jac', 'cl', 'cu'])
            # constraints is provided as a dict, supported by trust-constr and Ipopt
            self.cl = constraint['cl']
            self.cu = constraint['cu']
        else:
            raise ValueError("constraints must be provided as a dict of a list of dict.")
        self.ncon = len(self.cl)
        self.nlp = cyipopt.Problem(
                            n=self.nvar,
                            m=self.ncon,
                            problem_obj=self,
                            lb=self.xl,
                            ub=self.xu,
                            cl=self.cl,
                            cu=self.cu
                        )

    def objective(self, x):
        return self.eval_f(x)

    def gradient(self, x):
        return self.eval_g(x)

    def constraints(self, x):
        if isinstance(self.cons, list):
            return np.array([con['fun'](x) for con in self.cons])
        else:
            return self.cons['cons'](x)

    def jacobian(self, x):
        if isinstance(self.cons, list):
            jacs = []
            for con in self.cons:
                if 'jac' in con:
                    jacs.append(con['jac'](x))
                else:
                    raise ValueError("Jacobian not provided for constraint.")
            return np.vstack(jacs)
        else:
            return self.cons['jac'](x)

    def solve(self, x0, solver_options=None):
        ipopt_options = self.ipopt_options
        if solver_options is not None:
            ipopt_options = solver_options
        if ipopt_options is not None:
            for key, value in ipopt_options.items():
                self.nlp.add_option(key, value)

        # Solve the optimization problem
        return self.nlp.solve(x0)


#### BO Base

In [7]:
# A base class defining a general framework for Bayesian Optimization
class BOAlgorithmBase:
    def __init__(self):
        self.acquisition_type = "LCB" # Type of acquisition function (default = "LCB")
        self.batch_type = "KB"        # strategy for qEI
        self.xtrain = None            # Training data
        self.ytrain = None            # Training data
        self.prob   = None            # Problem structure
        self.evaluator = Evaluator()  # compute control objective evaluations
        self.bo_maxiter = 20          # Maximum number of Bayesian optimization steps
        self.n_start = 10             # estimating acquisition global optima by determining local optima n_start times and then determining the discrete max of that set
        self.batch_size = 1           # batch size
        # save some internal member train
        self.y_hist = None            # History of evaluations
        self.x_hist = None            # History of evaluations
        self.x_opt = None             # Best observed point
        self.y_opt = None             # Best observed value
        self.idx_opt = None           # Index of the best observed value in the history

    # Sets the acquisition function type and batch size
    def setAcquisitionType(self, acquisition_type, batch_size=1):
        self.acquisition_type = acquisition_type
        self.batch_size = batch_size

    # Sets the training data
    def setTrainingData(self, xtrain, ytrain):
        self.xtrain = xtrain
        self.ytrain = ytrain

    # Method to perform Bayesian optimization
    def optimize(self, fun):
        assert NotImplementedError("Child class of hiopEGO should implement method optimize")

    # Method to return the recorded optimization iterations and objectives
    def getOptimizationHistory(self):
        x_hist = np.array(self.x_hist, copy=True)
        y_hist = np.array(self.y_hist, copy=True)
        return x_hist, y_hist

    # Method to return the optimal solution 
    def getOptimalPoint(self):
        x_opt = np.array(self.x_opt, copy=True)
        return x_opt

    # Method to return the optimal objective
    def getOptimalObjective(self):
        y_opt = np.array(self.y_opt, copy=True)
        return y_opt



#### BO Algorithm

In [8]:
# A subclass of BOAlgorithmBase implementing a full Bayesian Optimization workflow
class BOAlgorithm(BOAlgorithmBase):
    def __init__(self, prob:Problem, gpsurrogate:GaussianProcess, xtrain, ytrain,
                 user_grad = None,
                 options = {}):
        super().__init__()
        
        assert isinstance(gpsurrogate, GaussianProcess)
        
        self.setTrainingData(xtrain, ytrain)
        self.prob = prob
        self.gpsurrogate = gpsurrogate
        self.bounds = self.gpsurrogate.get_bounds()
        self.fun_grad = None

        self.bo_maxiter = options.get('bo_maxiter', self.bo_maxiter)
        assert self.bo_maxiter > 0, f"Invalid bo_maxiter: {self.bo_maxiter }"
        
        self.solver_options = {"maxiter": 200}
        self.solver_options = options.get('solver_options', self.solver_options)

        acquisition_type = options.get('acquisition_type', "LCB")
        assert acquisition_type in ["LCB", "EI"], f"Invalid acquisition_type: {acquisition_type}"

        batch_size = options.get('batch_size', 1)
        assert isinstance(batch_size, int), f"batch_size {batch_size} not an integer"
        assert batch_size > 0, f"batch_size {batch_size} is not strictly positive"
        self.setAcquisitionType(acquisition_type, batch_size)

        self.evaluator = options.get('evaluator', self.evaluator)
        assert isinstance(self.evaluator, Evaluator)

        acqf_method = options.get('acquisition_method', "multi_start") 
        assert acqf_method in ["multi_start", "bnb"], f"Invalid acqf_method: {acqf_method}"
        self.acqf_method = acqf_method

        if options and 'opt_solver' in options:
            opt_solver = options['opt_solver']
            assert opt_solver in ["SLSQP", "trust-constr", "IPOPT"], f"Invalid opt_solver: {opt_solver}"
        else:
            opt_solver = "SLSQP"

        if isinstance(prob.constraints, dict):
            assert opt_solver in ["trust-constr", "IPOPT"], f"Invalid opt_solver: {opt_solver} while constraints are defined as a dict"
        elif isinstance(prob.constraints, list):
            assert opt_solver in ["SLSQP", "IPOPT"], f"Invalid opt_solver: {opt_solver} while constraints are defined as a list of dict"

        self.set_method(opt_solver)

        if user_grad:
            self.fun_grad = user_grad


    # Method to set up a callback function to minimize the acquisition function
    def _setup_acqf_minimizer_callback(self):
        self.acqf_minimizer_callback = lambda fun, x0: minimizer(fun, x0, self.opt_solver, self.bounds, self.prob.constraints, self.solver_options)

    # Method to train the GP model
    def _train_surrogate(self, x_train, y_train):
        self.gpsurrogate.train(x_train, y_train)

    # Method to find the best next sampling point via optimizing the acquisition function
    def _find_best_point(self, x_train, y_train, x0 = None):

        self._train_surrogate(x_train, y_train)

        if self.acquisition_type == "LCB": #
            acqf = LCBacquisition(self.gpsurrogate)
            self.beta = acqf.beta   

        elif self.acquisition_type == "EI":
            acqf = EIacquisition(self.gpsurrogate)
        else:
            raise NotImplementedError("No implemented acquisition_type associated to" + +self.acquisition_type)

        acqf_obj_callback = lambda x: float(np.array(acqf.evaluate(np.atleast_2d(x))).flat[0])
        acqf_callback = {'obj': acqf_obj_callback}
        
        if acqf.has_gradient == True:
            acqf_grad_callback = lambda x: np.array(acqf.eval_g(np.atleast_2d(x)))
            acqf_callback['grad'] = acqf_grad_callback

        if self.acqf_method == "multi_start":
        
            x_all = []
            y_all = []

            for ii in range(self.n_start):
                success = False
                # Generate random starting point if x0 is not provided
                if self.prob is not None:
                    x0 = self.prob.sample(1)[0]
                else:
                    x0 = np.array([uniform(b[0], b[1]) for b in self.bounds])

                xopt, yout, success = self.acqf_minimizer_callback(acqf_callback, x0)

                if success:
                    x_all.append(xopt)
                    y_all.append(yout)

            if not x_all:

                raise RuntimeError("Optimization failed for all initial points — no solution found.")

            best_xopt = x_all[np.argmin(np.array(y_all))]

            return best_xopt
        
        elif self.acqf_method == "bnb":

            l_init = np.array([b[0] for b in self.bounds])
            u_init = np.array([b[1] for b in self.bounds])

            # Instantiate BnB with GP surrogate and BO callback
            bnb = BnBAlgorithm(
                x = x_train, 
                y = y_train,
                gpsurrogate=self.gpsurrogate,
                acqf_minimizer_callback=self.acqf_minimizer_callback, acquisition_type=self.acquisition_type,
            )

            # Run BnB optimization
            best_l, best_u, _ = bnb.optimize(l_init, u_init)

            # Take the midpoint of the best box as the candidate point
            x_best = 0.5 * (best_l + best_u)
            return x_best
            
    def _get_virtual_point(self, x):

        if self.batch_type not in ["CLmin", "KB", "KBUB", "KBLB", "KBRand"]:
            raise NotImplementedError("No implemented batch_type associated to"+self.batch_type)
        
        # constant-liar, Kriging-believer and Kriging-believer variants
        if self.batch_type == "CLmin":
            return min(self.gpsurrogate.training_y)
        elif self.batch_type == "KB":
            beta = 0.
        elif self.batch_type == "KBUB":
            beta = 3.0
        elif self.batch_type == "KBLB":
            beta = -3.0
        elif self.batch_type == "KBRand":
            beta = np.random.randn()
        return self.gpsurrogate.mean(x) + beta * np.sqrt(self.gpsurrogate.variance(x))

    # Set the optimization method
    def set_method(self, method):
        self.opt_solver = method

    # Set the options for the internal optimization solver
    def set_options(self, solver_options):
        self.solver_options = solver_options

    # Method to perform Bayesian optimization
    def optimize(self):
      x_train = self.xtrain
      y_train = self.ytrain
      
      n_init_sample = np.size(x_train, 0)
      self._setup_acqf_minimizer_callback()

      self.x_hist = []
      self.y_hist = []

      for i in range(self.bo_maxiter):
          print(f"*****************************")
          print(f"Iteration {i+1}/{self.bo_maxiter}")

          y_train_virtual = y_train.copy() # old training + batch_size num of virtual points
          for j in range(self.batch_size):
             # Get a new sample point
             x_new = self._find_best_point(x_train, y_train_virtual)
             
             # Update training sample points
             x_train         = np.vstack([x_train,         x_new    ])

             # if this is not the last point in the current batch
             # then obtain a virtual point
             if j < max(range(self.batch_size)):
                 # Get a virtual point
                 y_virtual = self._get_virtual_point(np.atleast_2d(x_new))

                 # Update training set with the virtual point
                 y_train_virtual = np.vstack([y_train_virtual, y_virtual])
          
          y_new = self.evaluator.run(self.prob.evaluate, x_train[-self.batch_size:])
          y_train = np.vstack([y_train, y_new])
          #print(f'x_new = {x_new}')
          #print(f'y_new = {y_new}')
          
          # Save the new sample points and objective evaluations
          for j in range(1, self.batch_size+1):
              self.x_hist.append(x_train[-j].flatten())
              self.y_hist.append(y_train[-j].flatten())
          if self.batch_size == 1:
              print(f"Sample point X: {x_train[-self.batch_size:]}, Observation Y: {y_new}")
          else:
              print(f"Sample points X: {x_train[-self.batch_size:]}, Observations Y: {y_new}")


      # Save the optimal results and all the training data
      self.idx_opt = np.argmin(self.y_hist)
      self.x_opt = self.x_hist[self.idx_opt]
      self.y_opt = self.y_hist[self.idx_opt]
      self.setTrainingData(x_train, y_train)

      print(f"\n\nOptimal at BO iteration: {self.idx_opt+1} ")
      #if self.idx_opt < n_init_sample:
      #    print(f"Optimal at initial sample: {self.idx_opt+1}")
      #else:
      #    print(f"Optimal at BO iteration: {self.idx_opt-n_init_sample+1} ")
          
      print(f"Optimal point: {self.x_opt.flatten()}, Optimal value: {self.y_opt}")
      print()
    
def minimizer(fun, x0, method, bounds, constraints, solver_options):
    if method == "SLSQP":
        if 'grad' in fun:
            y = minimize(fun['obj'], x0, method=method, bounds=bounds, jac=fun['grad'], constraints=constraints, options=solver_options)
        else:
            y = minimize(fun['obj'], x0, method=method, bounds=bounds, constraints=constraints, options=solver_options)
        success = y.success
        if not success:
            print(y.message)
        xopt = y.x
        yopt = y.fun
    elif method == "trust-constr":
        nonlinear_constraint = NonlinearConstraint(constraints['cons'], constraints['cl'], constraints['cu'], jac=constraints['jac'])
        y = minimize(fun['obj'], x0, method=method, bounds=bounds, constraints=[nonlinear_constraint], options=solver_options)
        success = y.success
        if not success:
            print(y.message)
        xopt = y.x
        yopt = y.fun
    else:
        ipopt_prob = IpoptProb(fun['obj'], fun['grad'], constraints, bounds, solver_options)
        print(x0)
        sol, info = ipopt_prob.solve(x0)

        status = info.get('status', -999)
        msg = info.get('status_msg', b'unknown error')
        if status == 0:
            # ipopt returns 0 as success
            success = True
        else:
            warnings.warn(f"Ipopt failed to solve the problem. Status msg: {msg}")
            success = False

        yopt = info['obj_val']
        xopt = sol

    return xopt, yopt, success

#### BnB Base

In [9]:
import numpy as np
import cvxpy as cp
from scipy import linalg

class BnBAlgorithmBase:
    def __init__(self, x = None, y = None):
        # Node class for priority queue
        self.BnBNode = BnBNode
        self.BnB_LBmethod = "IPOPT"
        #self.BnB_LBmethod = None  # Use CVXPY for lower bounds

        # Stopping criteria
        self.epsilon_gap = 1e-3
        self.epsilon_diam = 1e-2

        # Kernel info for bounds
        self.kernel_spec = None
        self.kernel_func = None
        self.y_min = None

        # Evaluation parameters
        self.theta = None
        self.ell = None  # ARD scaling for distance

        # Test
        self.enable_debug_checks = False  

        # BnB search state
        self.best_l = None
        self.best_u = None
        self.upper_bound = np.inf
        self.final_gap = None
        self.final_diameter = None
        self.total_nodes = 0
        self.verbose = False

        # Training data
        self.x = x
        self.y = y

    def sync_from_smt(self):
        
        sm = self.gpsurrogate.surrogatesmt
        par = sm.optimal_par

        # --- kernel / corr selection ---
        corr = sm.options["corr"]  # e.g., 'squar_exp', 'pow_exp', 'abs_exp', 'matern32', 'matern52'

        if corr == "pow_exp":
            # OptionsDictionary -> use membership + indexing (no .get)
            p = float(sm.options["pow_exp_power"]) if "pow_exp_power" in sm.options else 2.0
            if p not in (1.0, 2.0):
                # tighten if your 1D bound code only supports p in {1,2}
                raise ValueError("Single-d bounds support pow_exp only for p=1 or p=2")
            self.kernel_spec = "pow_exp"
            self.p = p
        elif corr == "squar_exp":
            # Gaussian is pow_exp with p=2
            self.kernel_spec = "pow_exp"
            self.p = 2.0
        elif corr == "abs_exp":
            # Exponential is pow_exp with p=1
            self.kernel_spec = "pow_exp"
            self.p = 1.0
        elif corr == "matern32":
            self.kernel_spec = "matern32"
        elif corr == "matern52":
            self.kernel_spec = "matern52"
        else:
            raise ValueError(f"Unsupported SMT corr '{corr}'")

        # --- hyperparameters / scaling ---
        theta = getattr(sm, "optimal_theta", None)
        if theta is None:
            theta = sm.corr.theta
        self.theta = np.asarray(theta, float).ravel()
        
        # For LCB, need to eventually pull this from the BO Acquisition Function class
        self.beta = 3.0

        self.X_offset, self.X_scale = sm.X_offset, sm.X_scale
        self.Xc = (self.x - self.X_offset) / self.X_scale
        self._normalize = lambda x: (np.asarray(x, float) - self.X_offset) / self.X_scale

        y_mean = getattr(sm, "y_mean", None)
        y_std  = getattr(sm, "y_std",  None)
        if y_mean is None or y_std is None:
            # fall back to training y if available
            if self.y is not None:
                y_mean = float(np.mean(self.y))
                y_std  = float(np.std(self.y))
            else:
                y_mean, y_std = 0.0, 1.0
        if not np.isfinite(y_std) or abs(y_std) < 1e-15:
            y_std = 1.0  # avoid degenerate scaling
        self.y_mean = float(y_mean)
        self.y_std  = float(y_std)
        
        # SMT uses a regression trend option ('poly'); enforce constant mean for your μ-bounds
        poly = sm.options["poly"] if "poly" in sm.options else "constant"
        if poly != "constant":
            raise NotImplementedError("μ-bounds assume poly='constant'")

        # params from training
        self.beta0 = float(np.asarray(par["beta"]).ravel()[0])
        self.gamma = np.asarray(par["gamma"], float).ravel()

        self.C = par["C"]
        self.sigma2 = np.asarray(par.get("sigma2", 1.0), float).reshape(()).item()
        self.sigma2_ri = float(par["sigma2_ri"] if "sigma2_ri" in par else self.sigma2)

        self._solve_C = lambda v: linalg.solve_triangular(self.C, v, lower=True)
    

    def set_kernel(self, kernel_spec):

        assert kernel_spec in ["abs_exp", "pow_exp", "matern32", "matern52"]
        self.kernel_spec = kernel_spec

        if kernel_spec == "abs_exp":  # ν = 1/2
            self.kernel_func = lambda d: np.exp(-np.sqrt(d))

        elif kernel_spec == "pow_exp":  # SE, ν = ∞
            self.kernel_func = lambda d: np.exp(-d)

        elif kernel_spec == "matern32":  # ν = 3/2
            self.kernel_func = lambda d: (
                (1 + np.sqrt(3) * np.sqrt(d)) *
                np.exp(-np.sqrt(3) * np.sqrt(d))
            )

        elif kernel_spec == "matern52":  # ν = 5/2
            self.kernel_func = lambda d: (
                (1 + np.sqrt(5) * np.sqrt(d) + (5/3) * d) *
                np.exp(-np.sqrt(5) * np.sqrt(d))
            )
    
    def ker_bounds(self, l, u):
        
        """
        Tight monotone bounds for k(x, X_i) over box [l,u] (original units).
        Returns kL, kU of shape (nt,), consistent with SMT’s kernels.
        """
        
        # normalize the box
        l_c = self._normalize(l).ravel()
        u_c = self._normalize(u).ravel()

        Xc  = self.Xc                # (nt, d)
        th  = self.theta.ravel()     # (d,)
        spec = self.kernel_spec

        # per-point, per-dimension distance extremes (normalized space)
        dmin = np.maximum(0.0, np.maximum(l_c - Xc, Xc - u_c))        # (nt,d)
        dmax = np.maximum(np.abs(l_c - Xc), np.abs(u_c - Xc))         # (nt,d)

        if spec == "pow_exp":
            # power-exponential: k = exp(-sum_j θ_j |dx_j|^p)
            p = getattr(self, "p", 2.0)
            s_min = (th * (dmin ** p)).sum(axis=1)
            s_max = (th * (dmax ** p)).sum(axis=1)
            kU = np.exp(-s_min)                                       # max on box
            kL = np.exp(-s_max)                                       # min on box

        elif spec == "matern12":
            # Matérn ν=1/2 (a.k.a. abs-exp): k = exp(-sum_j θ_j |dx_j|)
            s_min = (th * dmin).sum(axis=1)
            s_max = (th * dmax).sum(axis=1)
            kU = np.exp(-s_min)
            kL = np.exp(-s_max)

        elif spec == "matern32":
            # SMT separable form: ∏_j (1 + √3 θ_j |dx_j|) exp(-√3 θ_j |dx_j|)
            a = np.sqrt(3.0) * th
            gmin = (1 + a * dmin) * np.exp(-a * dmin)
            gmax = (1 + a * dmax) * np.exp(-a * dmax)
            kU = np.prod(gmin, axis=1)
            kL = np.prod(gmax, axis=1)

        elif spec == "matern52":
            # SMT separable form: ∏_j (1 + √5 θ_j |dx_j| + (5/3) θ_j^2 dx_j^2) exp(-√5 θ_j |dx_j|)
            b = np.sqrt(5.0) * th
            btmin, btmax = b * dmin, b * dmax
            gmin = (1 + btmin + (btmin**2)/3.0) * np.exp(-btmin)
            gmax = (1 + btmax + (btmax**2)/3.0) * np.exp(-btmax)
            kU = np.prod(gmin, axis=1)
            kL = np.prod(gmax, axis=1)

        else:
            raise ValueError(f"Unsupported kernel_spec: {spec}")

        return kL, kU

    def mu_bounds(self, kL, kU):
       
        # compute in normalized y-space
        lo = np.where(self.gamma >= 0.0, kL, kU)
        hi = np.where(self.gamma >= 0.0, kU, kL)
        mu_L_n = self.beta0 + float(np.dot(self.gamma, lo))  # normalized
        mu_U_n = self.beta0 + float(np.dot(self.gamma, hi))  # normalized

        # de-normalize like SMT
        mu_L = self.y_mean + self.y_std * mu_L_n
        mu_U = self.y_mean + self.y_std * mu_U_n
        return mu_L, mu_U

    def sigma2_bounds(self, kL, kU, lb_passes=2, clip_nonneg=True):
        
        """
        Variance bounds over r ∈ [kL,kU] for SMT KRG (poly='constant'),
        returned in ORIGINAL y-units (σ^2 * y_std^2).
        """
        
        kL = np.asarray(kL, float).ravel()
        kU = np.asarray(kU, float).ravel()
        n  = kL.size
        assert hasattr(self, "C") and hasattr(self, "sigma2"), "Call sync_from_smt() first"
        assert kL.shape == kU.shape == (n,) and np.all(kL <= kU), "bad kL/kU"

        C      = self.C
        sigma2 = float(self.sigma2)  # normalized-y process variance
        ones   = np.ones(n)

        from scipy import linalg
        tmp   = linalg.solve_triangular(C, ones, lower=True)    # C tmp = 1
        a_vec = linalg.solve_triangular(C.T, tmp,  lower=False) # C^T a = tmp
        S     = float(ones @ a_vec)

        def bracket(r):
            r  = np.asarray(r, float).ravel()
            rt = linalg.solve_triangular(C, r, lower=True)      # C^{-1} r
            tau = float(a_vec @ r)
            return 1.0 - float(rt @ rt) + (1.0 - tau)**2 / S

        # ----- UPPER bound via convex relaxation in z with r = C z (drop -||z||^2) -----
        import cvxpy as cp
        A = C.T @ a_vec4
        Q = (2.0 / S) * np.outer(A, A)   # PSD
        b = (-2.0 / S) * A

        z = cp.Variable(n)
        cons = [C @ z >= kL, C @ z <= kU]
        obj  = 0.5 * cp.quad_form(z, Q) + b @ z
        cp.Problem(cp.Minimize(obj), cons).solve(solver="OSQP")

        r_ub = (C @ z.value).reshape(-1)
        tau  = float(A @ z.value)
        f_ub = (1.0 + 1.0/S) + (tau**2)/S - (2.0/S)*tau  # UB after dropping -||z||^2
        if clip_nonneg: f_ub = max(f_ub, 0.0)
        s2_U_n = sigma2 * f_ub  # normalized-y variance UB

        # ----- LOWER bound: tiny coordinate descent on r-box (optional) -----
        if self.BnB_LBmethod != "IPOPT":
            r = r_ub.copy()
            f = bracket(r)
            for _ in range(max(0, int(lb_passes))):
                improved = False
                for i in range(n):
                    r_lo = r.copy(); r_lo[i] = kL[i]; f_lo = bracket(r_lo)
                    r_hi = r.copy(); r_hi[i] = kU[i]; f_hi = bracket(r_hi)
                    if f_lo + 1e-6 < f: r, f, improved = r_lo, f_lo, True
                    if f_hi + 1e-6 < f: r, f, improved = r_hi, f_hi, True
                if not improved: break
            if clip_nonneg: f = max(f, 0.0)
            s2_L_n = sigma2 * f
        else:
            s2_L_n = 0.0

        # de-normalize like SMT: σ^2_orig = y_std^2 * σ^2_norm
        ys2 = (self.y_std ** 2)
        return ys2 * s2_L_n, ys2 * s2_U_n

    def rs_ei(self, mu, sigma):

        y_min = np.min(self.y)

        if sigma > 1e-12:
            z = (y_min - mu) / sigma
            ei = (y_min - mu) * norm.cdf(z) + sigma * norm.pdf(z)
            return -ei
        else:
            # Deterministic case: EI = max(y_min - mu, 0)
            return -max(y_min - mu, 0.0)
    
    def rs_lcb(self, mu, sigma):

        return mu - self.beta * sigma


#### BnB Algorithm

In [10]:
import heapq
import collections
from scipy.stats import norm
from scipy.optimize import minimize
import itertools

class BnBNode:
    def __init__(self, l, u, aq_L, aq_U):
        self.l = l
        self.u = u
        self.aq_L = aq_L
        self.aq_U = aq_U
        self.diam = np.max(u - l)
        self.midpoint = 0.5 * (l + u)

    def __lt__(self, other):
        return self.aq_U > other.aq_U

class BnBAlgorithm(BnBAlgorithmBase):
    def __init__(self, x, y, gpsurrogate, acqf_minimizer_callback, acquisition_type):
        super().__init__(x = x, y =y)
        self.gpsurrogate = gpsurrogate
        self.acqf_minimizer_callback = acqf_minimizer_callback
        self.acquisition_type = acquisition_type
        
        self.sync_from_smt()

    def _branch(self, l, u):

        # Force to float to avoid truncation issues
        l = l.astype(float)
        u = u.astype(float)

        # Pick the dimension with largest length
        d = np.argmax(u - l)
        mid = 0.5 * (l[d] + u[d])

        # If the midpoint is the same as one bound (degenerate split), return nothing
        if np.isclose(mid, l[d]) or np.isclose(mid, u[d]):
            return []

        # Generate child boxes
        l1, u1 = l.copy(), u.copy()
        l2, u2 = l.copy(), u.copy()
        
        # Split along midpoint
        u1[d] = mid
        l2[d] = mid

        return [(l1, u1), (l2, u2)]


    # For minimization, we find a feasible function value as the upper bound on the minimum value of the acquisition function.
    def compute_acq_upper_bound(self, l, u):

            if self.BnB_LBmethod == "IPOPT":
                    
                    if self.acquisition_type == "LCB":

                        acqf = LCBacquisition(self.gpsurrogate)

                    elif self.acquisition_type == "EI":

                        acqf = EIacquisition(self.gpsurrogate)

                    else:
                        raise NotImplementedError("No implemented acquisition_type associated to" + self.acquisition_type)

                    acqf_obj_callback = lambda x: float(np.array(acqf.evaluate(np.atleast_2d(x))).flat[0])
                    acqf_callback = {'obj': acqf_obj_callback}
                    if acqf.has_gradient == True:
                                acqf_grad_callback = lambda x: np.array(acqf.eval_g(np.atleast_2d(x)))
                                acqf_callback['grad'] = acqf_grad_callback

                    #if x0 is None: #Need to fix this.
                    x0 = np.array(uniform(l, u))
                    
                    xopt, yout, success = self.acqf_minimizer_callback(acqf_callback, x0)

                    if not success:
                        raise RuntimeError("EI maximization failed")

                    return float(yout)
            else:
                
                # We compute the upper bound of the acquisition function based on bounds of the kernel, mu and sigma.
                
                # Compute the kernel bounds with given x
                kL, kU = self.ker_bounds(l, u)
                
                # Compute the mean bounds
                mu_L, mu_U = self.mu_bounds(kL, kU)
                var_L, var_U = self.sigma2_bounds(kL, kU)
                
                if self.acquisition_type == "LCB":

                    lcb_U = self.rs_lcb(mu_U, np.sqrt(var_L))
                    return lcb_U
                
                elif self.acquisition_type == "EI":
                     
                    ei_U = self.rs_ei(mu_U, np.sqrt(var_L))
                    return ei_U
                
                else:
                    raise NotImplementedError("No implemented acquisition_type associated to" + self.acquisition_type)

    # For minimization, we compute the lower bound explicitly using the acquisition function over mu, sigma.
    def compute_acq_lower_bound(self, l,u):

                # We compute the upper bound of the acquisition function based on bounds of the kernel, mu and sigma.
                
                # Compute the kernel bounds with given x
                kL, kU = self.ker_bounds(l, u)
                # Compute the mean bounds
                mu_L, mu_U = self.mu_bounds(kL, kU)
                var_L,var_U = self.sigma2_bounds(kL, kU)

                if self.enable_debug_checks:

                    l_check = np.asarray(l).reshape(-1)
                    u_check = np.asarray(u).reshape(-1)
                    c_check = 0.5*(l+u)

                    # center + for each axis i: set x_i to l_i and u_i, others at center
                    Xchk = [c_check]
                    for i in range(l.size):
                        x_lo = c_check.copy(); x_lo[i] = l_check[i]; Xchk.append(x_lo)
                        x_hi = c_check.copy(); x_hi[i] = u_check[i]; Xchk.append(x_hi)
                    Xchk = np.vstack(Xchk)  # shape (1+2d, d)

                    # GP mean at those points (vector length 1+2d)
                    mu_vec = np.asarray(self.gpsurrogate.mean(Xchk)).reshape(-1)
                    var_vec = np.asarray(self.gpsurrogate.variance(Xchk)).reshape(-1)

                    # check
                    tol = 1e-8
                    ok_mu = (mu_vec >= mu_L - tol) & (mu_vec <= mu_U + tol)
                    if not np.all(ok_mu):
                        bad = np.where(~ok_mu)[0].tolist()
                        print(f"[μ] bounds violated at indices {bad}: "
                            f"min={mu_vec.min():.6g}, max={mu_vec.max():.6g}, "
                            f"bounds=({mu_L:.6g},{mu_U:.6g})")
                    ok_var = (var_vec <= var_U + tol)

                    if not np.all(ok_var):
                        bad = np.where(~ok_var)[0].tolist()
                        print(f"[σ] bounds violated at indices {bad}: "
                            f"min={var_vec.min():.6g}, max={var_vec.max():.6g}, "
                            f"bounds=(0,{var_U:.6g})")

                if self.acquisition_type == "LCB":

                    lcb_U = self.rs_lcb(mu_L, np.sqrt(var_U))
                    return lcb_U
                
                elif self.acquisition_type == "EI":
                     
                    ei_U = self.rs_ei(mu_L, np.sqrt(var_U))
                    return ei_U
                
                else:
                    raise NotImplementedError("No implemented acquisition_type associated to" + self.acquisition_type)

    def _prune_queue(self, queue, gub, eps):
        """Keep only nodes that can beat current GUB within tolerance; then re-heapify."""
        # queue items are (L, counter, node)
        pruned = [(L, c, n) for (L, c, n) in queue if L < gub - eps]
        heapq.heapify(pruned)
        return pruned

    def optimize(self, l_init, u_init):
        """
        Branch & Bound minimization with tolerance stopping.
        Core logic only: correct heap order, pruning on GUB tightening,
        single global stop, diameter continue, consistent per-node prune.
        """
        print("=== Starting Branch & Bound Optimization (Minimization) ===")
        print(f"Initial bounds: l = {l_init}, u = {u_init}")
        print(f"Number of points: {self.x.shape[0]}, Dim = {self.x.shape[1]}")

        # Root bounds
        aq_L_val = self.compute_acq_lower_bound(l_init, u_init)
        aq_U_val = self.compute_acq_upper_bound(l_init, u_init)
        print(f"\nInitial acquisition lower bound: {aq_L_val}")
        print(f"Initial acquisition upper bound: {aq_U_val}")

        # Init root + heap ordered by aq_L
        root = BnBNode(l_init.astype(float), u_init.astype(float), aq_L_val, aq_U_val)
        
        # --- HEAP STORES TUPLES: (L, counter, node) ---
        self._ctr = getattr(self, "_ctr", itertools.count())
        queue = [(root.aq_L, next(self._ctr), root)]
        heapq.heapify(queue)

        # Global best (minimization GUB)
        self.best_val = aq_U_val
        self.best_l, self.best_u = l_init.copy(), u_init.copy()

        diameters = [float(np.max(u_init - l_init))]
        self.total_nodes = 1
        iteration = 0

        while queue:
            iteration += 1
            # pop the node with smallest L
            L_top, _, node = heapq.heappop(queue)

            # sanity: popped L must equal node.aq_L and be <= any remaining L
            assert abs(L_top - node.aq_L) <= 1e-12
            if queue:
                min_rest = min(L for (L,_,_) in queue)
                assert L_top <= min_rest + 1e-12, f"Heap not ordered by L (popped {L_top}, min_rest {min_rest})"

            # Update GUB
            if node.aq_U < self.best_val:
                self.best_val = node.aq_U
                self.best_l, self.best_u = node.l, node.u
                queue = self._prune_queue(queue, self.best_val, self.epsilon_gap)

            print(f"\n--- Iteration {iteration} ---")
            print(f"Node bounds: l={node.l}, u={node.u}")
            print(f"Node acquisition bounds: L={node.aq_L}, U={node.aq_U}")
            print(f"Current best feasible value (GUB): {self.best_val}")

            # Global stop: GUB - GLB <= eps  (GLB == node.aq_L == L_top)
            if self.best_val - node.aq_L <= self.epsilon_gap:
                print(f"STOP: GUB - Node.L = {self.best_val - node.aq_L} <= {self.epsilon_gap}")
                break

            # Diameter stop (node-local)
            node_diam = float(np.max(node.u - node.l))
            if node_diam <= self.epsilon_diam:
                print(f"Skip: Node diameter {node_diam} <= {self.epsilon_diam}")
                continue

            # Per-node prune (consistent with stop rule)
            if node.aq_L >= self.best_val - self.epsilon_gap:
                print("Pruned: Node cannot improve best within tolerance.")
                continue

            # --- Branch ---
            for l_child, u_child in self._branch(node.l, node.u):
                print(f"  Branching to child: l={l_child}, u={u_child}")

                aq_L_r = self.compute_acq_lower_bound(l_child, u_child)
                aq_U_r = self.compute_acq_upper_bound(l_child, u_child)
                print(f"  Child acquisition bounds: L={aq_L_r}, U={aq_U_r}")

                self.total_nodes += 1
                diameters.append(float(np.max(u_child - l_child)))

                # Update GUB from child and prune if improved
                if aq_U_r < self.best_val:
                    self.best_val = aq_U_r
                    self.best_l, self.best_u = l_child, u_child
                    queue = self._prune_queue(queue, self.best_val, self.epsilon_gap)

                # Child-level prune (same tolerance)
                if aq_L_r >= self.best_val - self.epsilon_gap:
                    print("  Child pruned: L ≥ GUB - eps.")
                    continue

                # PUSH AS TUPLE
                child = BnBNode(l_child, u_child, aq_L_r, aq_U_r)
                heapq.heappush(queue, (child.aq_L, next(self._ctr), child))

            if not queue:
                print("\nSTOP: Queue empty, no better nodes remain.")
                break

            # Optional visibility
            phi_GLB = min(L for (L,_,_) in queue)
            print(f"Queue size: {len(queue)} | GLB={phi_GLB} | GUB={self.best_val} | Gap={self.best_val - phi_GLB}")

        self.final_gap = self.best_val - min([L for (L,_,_) in queue], default=self.best_val)
        self.final_diameter = min(diameters) if diameters else float('inf')

        print("\n=== Optimization Finished ===")
        print(f"Total nodes explored: {self.total_nodes}")
        print(f"Best bounds: l={self.best_l}, u={self.best_u}")
        print(f"Best feasible acquisition value (GUB): {self.best_val}")
        print(f"Final gap: {self.final_gap}, final diameter: {self.final_diameter}")

        return self.best_l, self.best_u, self.best_val

#### Example 

In [11]:
# Get user input for the number of repetitions from command-line arguments
num_repeat = 1

### parameters
n_samples = 5  # number of the initial samples to train GP
theta = 1.e-2  # hyperparameter for GP kernel
nx = 2         # dimension of the problem
xlimits = np.array([[-5, 5], [-5, 5]]) # bounds on optimization variable

prob_type_l = ["LpNorm", "Branin"]
acq_type_l = ["LCB", "EI"]

prob_type_l = ["LpNorm"]
acq_type_l = ["LCB"]

def con_eq(x):
  return  x[0]**2 + x[1]**2 - 1

def con_jac_eq(x):
  return  np.array([2*x[0], 2*x[1]])

def con_ineq(x):
  return  x[0] - x[1]

def con_jac_ineq(x):
  return  np.array([1.0, -1.0])

# 'SLSQP' requires constraints defined in a list of dict
user_constraint_list = [{'type': 'eq','fun': con_eq,   'jac': con_jac_eq},
                   {'type': 'ineq', 'fun': con_ineq, 'jac': con_jac_ineq}]

def cons_vec(x):
    x1, x2 = x
    return np.array([
        (x1 - 2)**2 + (x2 - 2.5)**2 - 2,
        x1 + x2 - 5,
        -x1
    ])

# Jacobian of constraints
def cons_jac_vec(x):
    x1, x2 = x
    return np.array([
        [2 * (x1 - 2), 2 * (x2 - 2.5)],
        [1, 1],
        [-1, 0]
    ])

cl = -np.inf * np.ones(3)
cu = np.zeros(3)

# 'trust-constr' method supports vector-valued constraints
user_constraint_dict = {'cons': cons_vec, 'jac': cons_jac_vec, 'cl': cl, 'cu': cu}


In [12]:
retval = 0
for prob_type in prob_type_l:
   print()
   if prob_type == "LpNorm":
      problem = LpNormProblem(nx, xlimits)
   else:
      problem = BraninProblem()
   problem.set_constraints(user_constraint_dict)

   for acq_type in acq_type_l:
      print("Problem name: ", problem.name)
      print("Acquisition type: ", acq_type)

      ### initial training set
      x_train = problem.sample(n_samples)
      y_train = problem.evaluate(x_train)

      ### Define the GP surrogate model
      gp_model = smtKRG(theta, xlimits, nx)
      gp_model.train(x_train, y_train)

      print(f"GP model trained with theta = {theta}")

      options = {
        'acquisition_type': acq_type,
        'acquisition_method': 'bnb',
        'bo_maxiter': 10,
        'opt_solver': 'IPOPT', #"SLSQP" "IPOPT" "trust-constr"
        'solver_options': {
        #   'maxiter': 100 #,'print_level': 5
           }
      }
   
      # Instantiate and run Bayesian Optimization
      bo = BOAlgorithm(problem, gp_model, x_train, y_train, options = options) #EI or LCB
      bo.optimize()

#sys.exit(retval)



Problem name:  LpNormProblem
Acquisition type:  LCB
GP model trained with theta = 0.01
*****************************
Iteration 1/10
=== Starting Branch & Bound Optimization (Minimization) ===
Initial bounds: l = [-5 -5], u = [5 5]
Number of points: 5, Dim = 2


/var/folders/lj/r44wsqnj64d42xclq3w52d2w0000gn/T/ipykernel_29037/1531344113.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.y_mean = float(y_mean)
/var/folders/lj/r44wsqnj64d42xclq3w52d2w0000gn/T/ipykernel_29037/1531344113.py:97: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.y_std  = float(y_std)


NameError: name 'a_vec4' is not defined

In [ ]:
retval = 0
for prob_type in prob_type_l:
   print()
   if prob_type == "LpNorm":
      problem = LpNormProblem(nx, xlimits)
   else:
      problem = BraninProblem()
   problem.set_constraints(user_constraint_dict)

   for acq_type in acq_type_l:
      print("Problem name: ", problem.name)
      print("Acquisition type: ", acq_type)

      ### initial training set
      x_train = problem.sample(n_samples)
      y_train = problem.evaluate(x_train)

      ### Define the GP surrogate model
      gp_model = smtKRG(theta, xlimits, nx)
      gp_model.train(x_train, y_train)

      options = {
        'acquisition_type': acq_type,
        'acquisition_method': 'multi_start', # "multi_start" "bnb"
        'bo_maxiter': 10,
        'opt_solver': 'IPOPT', #"SLSQP" "IPOPT" "trust-constr"
        'solver_options': {
         'print_level': 0
           }
      }
   
      # Instantiate and run Bayesian Optimization
      bo = BOAlgorithm(problem, gp_model, x_train, y_train, options = options) #EI or LCB
      bo.optimize()

#sys.exit(retval)



Problem name:  LpNormProblem
Acquisition type:  LCB
*****************************
Iteration 1/10
[-4.80160625 -4.26934347]

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

[4.46075923 3.41823702]
[-3.298709    0.99440893]
[4.95246267 0.11839221]
[-1.97954132 -1.2085479 ]
[-3.58669357  2.48917065]
[ 1.89687873 -3.95491584]
[-2.93342714 -4.51339412]
[-4.51713689 -1.20661974]
[2.56702794 0.80248562]
Sample point X: [[0.58578643 2.50003692]], Observation Y: [[2.56774811]]
*****************************
Iteration 2/10
[3.49786856 2.70125803]
[-4.52839174 -1.85128993]
[ 2.98029144 -2.30975349]
[4.20560438 0.55839138]
[-2.28484236 -4.13043031]
[-4.98164247 -4.6

In [ ]:
import numpy as np
from dataclasses import dataclass

import numpy as np
from scipy.stats import qmc  # already used by Problem

# Get user input for the number of repetitions from command-line arguments
num_repeat = 1

### parameters
n_samples = 5  # number of the initial samples to train GP
theta = 1.e-2  # hyperparameter for GP kernel
nx = 2         # dimension of the problem
xlimits = np.array([[-5, 5], [-5, 5]]) # bounds on optimization variable

class QuadraticShift2D(Problem):
    """
    f(x) = ||x - c||^2
    - Global minimizer: x* = c
    - Global minimum:   f* = 0
    """
    def __init__(self, xlimits=None, c=None, constraints=[]):
        ndim = 2
        if xlimits is None:
            xlimits = np.array([[-5.0, 5.0], [-5.0, 5.0]], dtype=float)
        name = "QuadraticShift2D"
        super().__init__(ndim, xlimits, name=name, constraints=constraints)

        # choose center if not provided
        if c is None:
            c = self.xlimits.mean(axis=1)
        self.c = np.asarray(c, dtype=float)
        assert self.c.shape == (2,), "c must be a 2D point"

        # expose known solution for checking later
        self.x_star = self.c.copy()
        self.f_star = 0.0

    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        ne, nx = x.shape
        assert nx == self.ndim
        diff = x - self.c[None, :]
        y = np.sum(diff * diff, axis=1, dtype=float).reshape(ne, 1)
        return y


retval = 0


problem = QuadraticShift2D(xlimits=xlimits, c=np.array([1.25, -2.0]))
problem.set_constraints([])  
    
x_train = problem.sample(n_samples)
y_train = problem.evaluate(x_train)

gp_model = smtKRG(theta, xlimits, nx)
gp_model.train(x_train, y_train)

print(f"GP model trained with theta = {theta}")

print("Compute the max and min distance using the d norm:")

print("x corresponding to dL:")

print("x corresponding to dU:")

print("Compute the variance on x corresponding to dL and dU with SMT:")

print("Compute the mean on x corresponding to dL and dU with SMT:")

print("Compute the variance on x corresponding to dL and dU with SMT:")

print("Compute the mean on x corresponding to dL and dU with SMT:")

GP model trained with theta = 0.01
Compute the max and min distance using the d norm:
x corresponding to dL:
x corresponding to dU:
Compute the variance on x corresponding to dL and dU with SMT:
Compute the mean on x corresponding to dL and dU with SMT:
Compute the variance on x corresponding to dL and dU with SMT:
Compute the mean on x corresponding to dL and dU with SMT:


In [ ]:
# ---- Use only BnBAlgorithmBase methods + SMT predict_* for reference ----

sm = gp_model.surrogatesmt

# Build a base "probe" and populate it from the trained SMT model
base = BnBAlgorithmBase(x=x_train, y=y_train)
base.gpsurrogate = gp_model
base.sync_from_smt()              # fills kernel_spec/p, theta, Xc, offsets, beta0/gamma, C, sigma2

# (Optional) ensure we try to compute a nontrivial lower bound for variance
base.BnB_LBmethod = None          # if "IPOPT", sigma2_L will be 0 in your current code

# Pick a box to test (use full domain here; swap l/u to any node box you want)
l = xlimits[:, 0].astype(float)
u = xlimits[:, 1].astype(float)

# 1) Bounds from your base class
kL, kU         = base.ker_bounds(l, u)       # per-point kernel bounds vs each training sample
mu_L, mu_U     = base.mu_bounds(kL, kU)      # scalar μ bounds on the box
s2_L, s2_U     = base.sigma2_bounds(kL, kU)  # scalar σ² bounds on the box

print("\n--- BnB base bounds on box ---")
print(f"mu_L={mu_L:.9f}, mu_U={mu_U:.9f}")
print(f"s2_L={s2_L:.9e}, s2_U={s2_U:.9e}")

# 2) (Optional) Node LCB/EI bounds using ONLY base.rs_* with your bounds
beta = 3.0  # or whatever you're using in BO; set from outside as needed
LCB_L = base.rs_lcb(mu_L, np.sqrt(max(s2_U, 0.0)))     # lower bound on LCB over the box
LCB_U = base.rs_lcb(mu_U, np.sqrt(max(s2_L, 0.0)))     # upper bound on LCB over the box
print(f"LCB node bounds:  L={LCB_L:.9f}   U={LCB_U:.9f}")

# 3) (Optional) Quick SMT sanity on a few points INSIDE the same box
#    (purely for reference; still using SMT’s own predict_* functions)
P = np.vstack([
    l,
    u,
    0.5*(l+u),
    np.array([l[0], u[1]]),
    np.array([u[0], l[1]]),
])
mu_smt  = sm.predict_values(P).ravel()
var_smt = sm.predict_variances(P).ravel()
print("\n--- SMT reference on corners/center ---")
print("μ(P):", mu_smt)
print("σ²(P):", var_smt)
print(f"min μ(P)={mu_smt.min():.9f}, max μ(P)={mu_smt.max():.9f}")
print(f"min σ²(P)={var_smt.min():.9e}, max σ²(P)={var_smt.max():.9e}")

# 4) (Optional) show per-training-point kernel extremes match your ker_bounds
#    Build the witness x for each training point that attains kU_i and kL_i.
#    (Just geometry: clamp for kU, farther endpoint for kL.)
off, scl = base.X_offset, base.X_scale
Xc = base.Xc
l_c = (l - off) / scl
u_c = (u - off) / scl

def extreme_points_for_training_point(Xi_c, l_c, u_c):
    xU_c = np.clip(Xi_c, l_c, u_c)                # min distance → max kernel
    dL = np.abs(l_c - Xi_c); dU = np.abs(u_c - Xi_c)
    xL_c = np.where(dL >= dU, l_c, u_c)           # max distance → min kernel
    return xU_c, xL_c

# compute kernel at those witnesses via base.ker_bounds result for equality check
kU_w, kL_w = [], []
for i in range(Xc.shape[0]):
    xU_c, xL_c = extreme_points_for_training_point(Xc[i], l_c, u_c)
    # evaluate kernel component i at those x by reusing your distance→kernel mapping:
    # Use the same separable form your ker_bounds uses:
    # we’ll reconstruct this single component using the *same* logic as ker_bounds,
    # but without re-deriving formulas: take the per-dim distances and plug into the same kernel_spec.
    dxU = np.abs(xU_c - Xc[i]); dxL = np.abs(xL_c - Xc[i])
    th  = base.theta
    if base.kernel_spec == "pow_exp":
        p = getattr(base, "p", 2.0)
        kU_w.append(np.exp(-np.dot(th, dxU**p)))
        kL_w.append(np.exp(-np.dot(th, dxL**p)))
    elif base.kernel_spec == "matern32":
        a = np.sqrt(3.0) * th
        kU_w.append(np.prod((1 + a*dxU) * np.exp(-a*dxU)))
        kL_w.append(np.prod((1 + a*dxL) * np.exp(-a*dxL)))
    elif base.kernel_spec == "matern52":
        b = np.sqrt(5.0) * th; tU=b*dxU; tL=b*dxL
        kU_w.append(np.prod((1 + tU + (tU**2)/3.0) * np.exp(-tU)))
        kL_w.append(np.prod((1 + tL + (tL**2)/3.0) * np.exp(-tL)))
    else:
        raise ValueError(f"Unsupported kernel_spec: {base.kernel_spec}")

kU_w = np.array(kU_w); kL_w = np.array(kL_w)
print("\nmax |kU(witness) - kU(ker_bounds)| =", float(np.max(np.abs(kU_w - kU))))
print("max |kL(witness) - kL(ker_bounds)| =", float(np.max(np.abs(kL_w - kL))))


Polishing not needed - no active set detected at optimal point

--- BnB base bounds on box ---
mu_L=0.135986001, mu_U=7.735911562
s2_L=0.000000000e+00, s2_U=1.428958646e+01
LCB node bounds:  L=-11.204484808   U=7.735911562

--- SMT reference on corners/center ---
μ(P): [4.80718608 4.51374837 0.59620484 3.94602162 3.92736686]
σ²(P): [2.38859383 2.99219868 0.12801777 4.18766475 4.20605514]
min μ(P)=0.596204841, max μ(P)=4.807186077
min σ²(P)=1.280177686e-01, max σ²(P)=4.206055140e+00

max |kU(witness) - kU(ker_bounds)| = 0.0
max |kL(witness) - kL(ker_bounds)| = 0.0


/var/folders/lj/r44wsqnj64d42xclq3w52d2w0000gn/T/ipykernel_31483/961351768.py:95: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.y_mean = float(y_mean)
/var/folders/lj/r44wsqnj64d42xclq3w52d2w0000gn/T/ipykernel_31483/961351768.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.y_std  = float(y_std)


In [ ]:
import numpy as np
from dataclasses import dataclass

import numpy as np
from scipy.stats import qmc  # already used by Problem

# Get user input for the number of repetitions from command-line arguments
num_repeat = 1

### parameters
n_samples = 5  # number of the initial samples to train GP
theta = 1.e-2  # hyperparameter for GP kernel
nx = 2         # dimension of the problem
xlimits = np.array([[-5, 5], [-5, 5]]) # bounds on optimization variable

class QuadraticShift2D(Problem):
    """
    f(x) = ||x - c||^2
    - Global minimizer: x* = c
    - Global minimum:   f* = 0
    """
    def __init__(self, xlimits=None, c=None, constraints=[]):
        ndim = 2
        if xlimits is None:
            xlimits = np.array([[-5.0, 5.0], [-5.0, 5.0]], dtype=float)
        name = "QuadraticShift2D"
        super().__init__(ndim, xlimits, name=name, constraints=constraints)

        # choose center if not provided
        if c is None:
            c = self.xlimits.mean(axis=1)
        self.c = np.asarray(c, dtype=float)
        assert self.c.shape == (2,), "c must be a 2D point"

        # expose known solution for checking later
        self.x_star = self.c.copy()
        self.f_star = 0.0

    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        ne, nx = x.shape
        assert nx == self.ndim
        diff = x - self.c[None, :]
        y = np.sum(diff * diff, axis=1, dtype=float).reshape(ne, 1)
        return y


prob_type_l = ["Quad2D"]   # simple known-minimizer test
acq_type_l  = ["EI"]

retval = 0
for prob_type in prob_type_l:
    print()
    if prob_type == "Quad2D":
        problem = QuadraticShift2D(xlimits=xlimits, c=np.array([1.25, -2.0]))
        problem.set_constraints([])  # <-- no constraints for this test
    elif prob_type == "LpNorm":
        problem = LpNormProblem(nx, xlimits)
        #problem.set_constraints(user_constraint_dict)  # only if you want constraints there

    for acq_type in acq_type_l:
        print("Problem name: ", problem.name)
        print("Acquisition type: ", acq_type)

        x_train = problem.sample(n_samples)
        y_train = problem.evaluate(x_train)

        gp_model = smtKRG(theta, xlimits, nx)
        gp_model.train(x_train, y_train)
        print(f"GP model trained with theta = {theta}")

        options = {
            'acquisition_type': acq_type,
            'acquisition_method': 'bnb',
            'bo_maxiter': 10,
            'opt_solver': 'IPOPT',
            'solver_options': {}
        }

        bo = BOAlgorithm(problem, gp_model, x_train, y_train, options=options)
        bo.optimize()

        if hasattr(bo, "best_x"):
            dist = np.linalg.norm(bo.best_x - problem.x_star)
            print(f"[Check] ||x_best - x*|| = {dist:.3e}  (x* = {problem.x_star})")